# 연습 문제 
- data/ratings_train.txt 로드 
- doc2vec를 이용하여 임베딩 데이터 생성 
- LogisticRegression, SVC 모델을 이용하여 감정 분석


In [56]:
# !pip install tqdm

In [57]:
import os, re, random
import numpy as np
import pandas as pd
# 진행 상황을 보여주는 라이브러리
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# 재현성
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


# KOMORAN
from konlpy.tag import Komoran
komoran = Komoran()  # 최초 생성 시 다소 시간이 걸릴 수 있습니다.

In [58]:
# 파일 형식: id \t document \t label
DATA_PATH = "data/ratings_train.txt"
assert os.path.exists(DATA_PATH), f"파일을 찾을 수 없습니다: {DATA_PATH}"

df = pd.read_csv(DATA_PATH, sep="\t")

print(df.shape)
df.head()


(150000, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [59]:
def normalize_text(text: str) -> str:
    # 특수문자 제거, 공백 정리
    text = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [60]:
# (1) 결측 제거
df = df.dropna(subset=["document", "label"]).reset_index(drop=True)


# (3) 텍스트 정규화 열 추가
df["document_norm"] = df["document"].apply(normalize_text)

# (4) 공백/빈 문자열 제거
df = df[df["document_norm"].str.len() > 1].reset_index(drop=True)

# (5) 중복 제거 (정규화된 텍스트 기준)
before = len(df)
df = df.drop_duplicates(subset=["document_norm"]).reset_index(drop=True)
after = len(df)
print(f"정규화/중복/공백 처리 후: {df.shape} (중복 제거: {before - after}건)")
df.head()

df = df.head(5000)

정규화/중복/공백 처리 후: (143379, 4) (중복 제거: 5313건)


In [61]:
# 간단 불용어 예시 (도메인에 맞게 확장 권장)
STOPWORDS = set(["하다", "되다", "이다", "것", "수", "거"])

# 사용할 품사 태그(주요 의미어 위주)
# NNG/NNP: 일반/고유 명사, VV: 동사 어간, VA: 형용사 어간, MAG: 일반 부사, XR: 어근
KEEP_POS = set(["NNG", "NNP", "VV", "VA", "MAG", "XR"])

def komoran_tokenize(text: str):
    """
    KOMORAN 기반 토크나이저.
    필요 시 불용어 처리/품사 필터링 등을 추가할 수 있습니다.
    """
    # 품사 기반 필터링
    pos = komoran.pos(text, join=False)
    tokens = []
    for morph, tag in pos:
        if tag in KEEP_POS:
            # VV/VA는 어간이므로 그대로 사용(필요시 '다' 붙이기 옵션 가능)
            # 예: if tag in ("VV","VA"): morph = morph + "다"
            if morph not in STOPWORDS and len(morph) > 1:
                tokens.append(morph)
    return tokens
def build_tagged_docs(norm_texts):
    tagged = []
    for i, t in enumerate(norm_texts):
        tokens = komoran_tokenize(t)
        if len(tokens) == 0:
            # 토큰이 전혀 없으면 빈 문서로 간주 → 제외
            continue
        tagged.append(TaggedDocument(words=tokens, tags=[i]))
    return tagged

# train/test split (ratings_train.txt만 사용 → 내부 분리)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label"]
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

tagged_train = build_tagged_docs(train_df["document_norm"].tolist())

print("train:", train_df.shape, "test:", test_df.shape)
print("Tagged sample:", tagged_train[0] if len(tagged_train) else "빈 태그")


train: (4000, 4) test: (1000, 4)
Tagged sample: TaggedDocument<['정말', '재미있', '음악', '템포', '편집', '기술', '두드러지', '작품'], [0]>


In [62]:
DOC2VEC_PARAMS = dict(
    vector_size=200,
    window=5,
    min_count=2,
    dm=1,              # 1: PV-DM, 0: PV-DBOW
    hs=0, negative=5,
    workers=4,
    seed=SEED,
    epochs=50
)

d2v = Doc2Vec(
    vector_size=DOC2VEC_PARAMS["vector_size"],
    window=DOC2VEC_PARAMS["window"],
    min_count=DOC2VEC_PARAMS["min_count"],
    dm=DOC2VEC_PARAMS["dm"],
    hs=DOC2VEC_PARAMS["hs"],
    negative=DOC2VEC_PARAMS["negative"],
    workers=DOC2VEC_PARAMS["workers"],
    seed=DOC2VEC_PARAMS["seed"],
    epochs=DOC2VEC_PARAMS["epochs"],
    dm_mean=1 if DOC2VEC_PARAMS["dm"] == 1 else 0  # PV-DM(mean)
)

d2v.build_vocab(tagged_train)
d2v.train(tagged_train, total_examples=len(tagged_train), epochs=DOC2VEC_PARAMS["epochs"])

print("Doc2Vec 학습 완료. vocab size:", len(d2v.wv))


Doc2Vec 학습 완료. vocab size: 2393


In [63]:
def infer_vectors(model: Doc2Vec, norm_texts, epochs=50, alpha=0.025):
    vecs = []
    # for t in tqdm(norm_texts, desc="Infer vectors"):
    for t in norm_texts:
        tokens = komoran_tokenize(t)
        if len(tokens) == 0:
            # 빈 토큰 문서는 영벡터/평균치 등으로 대체 가능. 여기선 영벡터로 채웁니다.
            vecs.append(np.zeros(model.vector_size, dtype=np.float32))
            continue
        v = model.infer_vector(tokens, epochs=epochs, alpha=alpha)
        vecs.append(v)
    return np.vstack(vecs)

X_train = infer_vectors(d2v, train_df["document_norm"].tolist(), epochs=50, alpha=0.025)
y_train = train_df["label"].to_numpy()

X_test  = infer_vectors(d2v, test_df["document_norm"].tolist(), epochs=50, alpha=0.025)
y_test  = test_df["label"].to_numpy()

X_train.shape, X_test.shape


((4000, 200), (1000, 200))

In [64]:
def eval_clf(X_tr, y_tr, X_te, y_te, clf, name):
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_te)
    acc = accuracy_score(y_te, pred)
    print(f"\n[{name}] Accuracy: {acc:.4f}")
    print(f"[{name}] Classification Report:\n{classification_report(y_te, pred, digits=4)}")
    print(f"[{name}] Confusion Matrix:\n{confusion_matrix(y_te, pred)}")

# 1) Logistic Regression
lr_clf = LogisticRegression(max_iter=2000, random_state=SEED)
eval_clf(X_train, y_train, X_test, y_test, lr_clf, "LogisticRegression")

# 2) Linear SVM
svm_clf = LinearSVC(random_state=SEED)
eval_clf(X_train, y_train, X_test, y_test, svm_clf, "LinearSVC")



[LogisticRegression] Accuracy: 0.7160
[LogisticRegression] Classification Report:
              precision    recall  f1-score   support

           0     0.7309    0.6873    0.7084       502
           1     0.7027    0.7450    0.7232       498

    accuracy                         0.7160      1000
   macro avg     0.7168    0.7161    0.7158      1000
weighted avg     0.7168    0.7160    0.7158      1000

[LogisticRegression] Confusion Matrix:
[[345 157]
 [127 371]]

[LinearSVC] Accuracy: 0.7200
[LinearSVC] Classification Report:
              precision    recall  f1-score   support

           0     0.7322    0.6972    0.7143       502
           1     0.7088    0.7430    0.7255       498

    accuracy                         0.7200      1000
   macro avg     0.7205    0.7201    0.7199      1000
weighted avg     0.7206    0.7200    0.7199      1000

[LinearSVC] Confusion Matrix:
[[350 152]
 [128 370]]
